## Finite Difference Methods

#### Import Required Libraries

In [2]:
# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Importing libraries
import pandas as pd
import numpy as np
from pathlib import Path
from tabulate import tabulate

# Import cufflinks
import cufflinks as cf
cf.set_config_file(offline=True, dimensions=((1000,600)))

# Set max row and columns to 300 and 100
pd.set_option('display.max_rows', 300)
pd.set_option('display.max_columns', 100)

#### Special Parameters

In [2]:
# Specify the parameters for FDM
T = 1                                           #time to maturity in years
E = 100                                         #strike price
r = .05                                         #riskfree rate
vol = .20                                       #volatility
Flag = 1                                        #flag = 1 for call, -1 for puts
EType = 1                                       #Exercise Type = 1 for american, 0 for european
NAS = 20                                        #number of asset steps

ds = 2 * E / NAS                                #asset step size
dt = (0.9/vol**2/NAS**2)                        #time step size, for stability

NTS  = int(T / dt) + 1                          #number of time steps
dt   = T / NTS                                  #time steps away [Expiration as int #of time steps away]

#### Generate Grid

In [3]:
# Create asset steps i*ds
s = np.arange(0, (NAS+1)*ds,ds)
s

array([  0.,  10.,  20.,  30.,  40.,  50.,  60.,  70.,  80.,  90., 100.,
       110., 120., 130., 140., 150., 160., 170., 180., 190., 200.])

In [4]:
# Create time steps k*dt
t = T-np.arange(NTS*dt,-dt,-dt)
t

array([0.        , 0.05555556, 0.11111111, 0.16666667, 0.22222222,
       0.27777778, 0.33333333, 0.38888889, 0.44444444, 0.5       ,
       0.55555556, 0.61111111, 0.66666667, 0.72222222, 0.77777778,
       0.83333333, 0.88888889, 0.94444444, 1.        ])

In [5]:
# Verify the steps size
s.shape, t.shape

((21,), (19,))

In [6]:
# Initialize the grid with zeros
grid = np.zeros((len(s),len(t)))
# Subsume the grid points into a dataframe
# with asset price as index and time steps as columns 
grid = pd.DataFrame(grid, index=s, columns=np.around(t,3)) 
grid

,0.000,0.056,0.111,0.167,0.222,0.278,0.333,0.389,0.444,0.500,0.556,0.611,0.667,0.722,0.778,0.833,0.889,0.944,1.000
0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
30.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
40.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
60.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
70.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
80.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
90.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


#### Set up Payoff

In [7]:
#Set Payoff at Expiration
if Flag == 1:
    grid.iloc[:,0] = np.maximum(s - E, 0)
else:
    grid.iloc[:,0] = np.maximum(E - s, 0)

In [8]:
#Verify the grid
grid

,0.000,0.056,0.111,0.167,0.222,0.278,0.333,0.389,0.444,0.500,0.556,0.611,0.667,0.722,0.778,0.833,0.889,0.944,1.000
0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
30.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
40.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
60.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
70.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
80.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
90.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
# Store payoff for early exercise
p = []
for j in np.arange(0, NAS+1):
    p.append(grid.iloc[j,0])

In [10]:
p

[0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 10.0,
 20.0,
 30.0,
 40.0,
 50.0,
 60.0,
 70.0,
 80.0,
 90.0,
 100.0]

#### Fill the Grid

In [11]:
# k is counter
for k in range(1, len(t)):
    for i in range(1,len(s)-1):
        delta = (grid.iloc[i+1,k-1] - grid.iloc[i-1,k-1]) / (2*ds)
        gamma = (grid.iloc[i+1,k-1]-2*grid.iloc[i,k-1]+grid.iloc[i-1,k-1]) / (ds**2)
        theta = (-0.5* vol**2 * s[i]**2 * gamma) - (r*s[i]*delta) + (r*grid.iloc[i,k-1])
        
        #Vnew = Vold - theta * dt
        grid.iloc[i,k] = grid.iloc[i,k-1] - (theta*dt)

     # Set boundary condition at S = 0
    grid.iloc[0,k] = grid.iloc[0,k-1] * (1-r*dt) # ds = rsdt + sigma*sdx, s= 0, ds = 0

    # Set boundary condition at S = infinity # gamma = 0, so you can linearly extract
    grid.iloc[len(s)-1,k] = 2*grid.iloc[len(s)-2,k] - grid.iloc[len(s)-3,k]
    
    
    if EType==1:
       for i in range(0,len(s)):
        grid.iloc[i,k] = np.maximum(grid.iloc[i,k], p[i])

# Round grid values to 2 decimals
grid = np.around(grid,3)

In [12]:
# Output the option values 
grid.iloc[10:20,:]
#grid

,0.000,0.056,0.111,0.167,0.222,0.278,0.333,0.389,0.444,0.500,0.556,0.611,0.667,0.722,0.778,0.833,0.889,0.944,1.000
100.0,0.0,1.250,2.253,3.093,3.819,4.466,5.054,5.599,6.109,6.593,7.054,7.497,7.925,8.339,8.741,9.133,9.515,9.889,10.255
110.0,10.0,10.278,10.671,11.118,11.587,12.063,12.535,13.000,13.455,13.901,14.337,14.763,15.180,15.589,15.990,16.383,16.770,17.149,17.523
120.0,20.0,20.278,20.555,20.848,21.159,21.486,21.826,22.174,22.529,22.887,23.247,23.607,23.967,24.326,24.683,25.038,25.391,25.741,26.089
130.0,30.0,30.278,30.555,30.831,31.109,31.392,31.680,31.973,32.272,32.575,32.882,33.192,33.504,33.819,34.134,34.451,34.768,35.085,35.402
140.0,40.0,40.278,40.555,40.831,41.106,41.382,41.658,41.935,42.213,42.494,42.777,43.061,43.348,43.636,43.926,44.217,44.509,44.802,45.095
150.0,50.0,50.278,50.555,50.831,51.106,51.381,51.655,51.929,52.202,52.476,52.750,53.025,53.300,53.576,53.853,54.130,54.408,54.687,54.966
160.0,60.0,60.278,60.555,60.831,61.106,61.381,61.655,61.928,62.201,62.473,62.745,63.016,63.287,63.558,63.829,64.100,64.372,64.643,64.914
170.0,70.0,70.278,70.555,70.831,71.106,71.381,71.655,71.928,72.201,72.472,72.743,73.014,73.284,73.553,73.822,74.091,74.359,74.627,74.895
180.0,80.0,80.278,80.555,80.831,81.106,81.381,81.655,81.928,82.201,82.472,82.743,83.014,83.283,83.552,83.820,84.088,84.355,84.621,84.887
190.0,90.0,90.278,90.555,90.831,91.106,91.381,91.655,91.928,92.201,92.472,92.743,93.013,93.283,93.552,93.819,94.087,94.353,94.619,94.883


In [13]:
# Print out stock, payoff and option value
data = {
    "Stock": s,
    "Payoff": p,
    "Option": grid.iloc[:,-1]
}
    
option_value_2D = pd.DataFrame(data)
option_value_2D # print(option_value_2D.to_string(index=False))

,Stock,Payoff,Option
0.0,0.0,0.0,0.000
10.0,10.0,0.0,0.000
20.0,20.0,0.0,0.000
30.0,30.0,0.0,0.000
40.0,40.0,0.0,0.000
50.0,50.0,0.0,0.006
60.0,60.0,0.0,0.066
70.0,70.0,0.0,0.430
80.0,80.0,0.0,1.754
90.0,90.0,0.0,4.899


In [14]:
# Plot option value and payoff
colo = ['blue', 'red']
option_value_2D[['Payoff', 'Option']].iplot(title='Payoff & Option Value', colors= colo)

## User Defined Function

In [15]:
def fdm_option(strike, sigma, rate, ttm, nas, is_call=True, american=False):
    # Specify flag as 1 for calls and -1 for puts
    # Specify American type option = True for early exercise and False for European
    ds = 2*strike/nas               #aseet step size
    dt = 0.9/sigma**2/nas**2        #for stability
    nts = int(ttm / dt) + 1         #time steps size, alternatively use fixed size 10 on stability issue
    dt = ttm/nts                    #time step
    
    s = np.arange(0,(nas+1)*ds,ds)
    t = ttm-np.arange(nts*dt,-dt,-dt)
   
   # Initialize the grid with zeros   
    grid = np.zeros((len(s),len(t)))
    grid = pd.DataFrame(grid, index=s, columns=np.around(t,2))

    # Set boundary condition at Expiration
    flag = 1 if is_call else -1
    grid.iloc[:,0] = abs(np.maximum(flag * (s - strike), 0))
    
    # Store payoff for early exercise
    p = []
    for j in range(0, nas+1):
        p.append(grid.iloc[j,0])
        
    for k in range(1, len(t)):
        for i in range(1,len(s)-1):
            delta = (grid.iloc[i+1,k-1] - grid.iloc[i-1,k-1]) / (2*ds)
            gamma = (grid.iloc[i+1,k-1]-2*grid.iloc[i,k-1]+grid.iloc[i-1,k-1]) /(ds**2)
            theta = (-0.5* sigma**2 * s[i]**2 * gamma) - (r*s[i]*delta) + (r*grid.iloc[i,k-1])
            grid.iloc[i,k] = grid.iloc[i,k-1] - dt*theta
            
        # Set boundary condition at S = 0
        grid.iloc[0,k] = grid.iloc[0,k-1] * (1-rate*dt)
    
        # Set boundary condition at S = infinity
        grid.iloc[len(s)-1,k] = abs(2*(grid.iloc[len(s)-2,k]) - grid.iloc[len(s)-3,k])
        
        # Check for early exercise
        if american==True:
            for i in range(0,len(s)):
                grid.iloc[i,k] = np.maximum(grid.iloc[i,k], p[i])
                
    # round grid values to 4 decimals
    return np.around(grid,2)

In [16]:
# Call the function to price options
fdm_grid = fdm_option(100,0.2,0.05,1,20)
fdm_grid

,0.00,0.06,0.11,0.17,0.22,0.28,0.33,0.39,0.44,0.50,0.56,0.61,0.67,0.72,0.78,0.83,0.89,0.94,1.00
0.0,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
10.0,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
20.0,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
30.0,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
40.0,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
50.0,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.01
60.0,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.01,0.01,0.02,0.02,0.03,0.04,0.05,0.07
70.0,0.0,0.00,0.00,0.00,0.00,0.00,0.01,0.02,0.03,0.04,0.07,0.09,0.13,0.16,0.21,0.26,0.31,0.37,0.43
80.0,0.0,0.00,0.00,0.01,0.04,0.08,0.14,0.22,0.31,0.42,0.53,0.66,0.80,0.94,1.10,1.25,1.42,1.58,1.75
90.0,0.0,0.00,0.13,0.34,0.59,0.88,1.18,1.49,1.81,2.13,2.45,2.77,3.08,3.39,3.70,4.00,4.31,4.60,4.90


### Option Values

In [17]:
# Output the option values European Call and put
callEuro = fdm_option(100,0.2,0.05,1,60,is_call=True, american=False).loc[100,1]
putEuro = fdm_option(100,0.2,0.05,1,60,is_call=False, american=False).loc[100,1]

callAmer = fdm_option(100,0.2,0.05,1,60,is_call=True, american=True).loc[100,1]
putAmer = fdm_option(100,0.2,0.05,1,60,is_call=False, american=True).loc[100,1]

# Print the values
header = ['American Call', 'American Put', 'European Call', 'European Put']
table = [[callAmer, putAmer, callEuro, putEuro]]

print(tabulate(table,header, numalign='center', floatfmt=("", ".5f")))

 American Call    American Put    European Call    European Put
---------------  --------------  ---------------  --------------
     10.43          6.07000           10.43            5.55


### Visualize the payoff

In [18]:
# Plot Option Payoff
# Axis titles are not rendered correctly on the graph. This is a bug in cufflinks 
fig = fdm_grid.iplot(kind = 'surface', title='Option values by Explicit FDM',xTitle='Spot', yTitle='Maturity', zTitle='Option Value', asFigure=True)
fig.show()

In [19]:
# Save the figure - you might have to install kaleido for earlier version of plotly
# Save as portable network graphics


# fig.write_image("images/fdm_option.png")

# cwd = Path.cwd()
# figfiles = cwd.parent/"figures"
# fig.write_image(figfiles/"fdm_option.png")

## Bilinear Interpolation

In [20]:
fdm_grid.columns[fdm_grid.columns < 0.25][-1]

0.22

In [21]:
def bilinear_interpolation(asset_price, ttm, df):
    
    # Find relevant rows and columns
    col1 = df.columns[df.columns < ttm][-1]
    col2 = df.columns[df.columns >= ttm][0]
    row1 = df.index[df.index < asset_price,][-1]
    row2 = df.index[df.index >= asset_price,][0]
    
    # Define points and areas
    V = [df.loc[row1, col1], df.loc[row1, col2],
         df.loc[row2, col2], df.loc[row2, col1]]
    A = [(row2 - asset_price) * (col2 - ttm),
         (row2 - asset_price) * (ttm - col1),
         (asset_price - row1) * (ttm - col1),
         (asset_price - row1) * (col2 - ttm)]
    
    # Interpolate values
    return sum(np.array(V)*np.array(A))/sum(np.array(A))

In [22]:
# Option value, approximated
bilinear_interpolation(105,0.25,fdm_grid)

7.984999999999999

In [23]:
# Verify rows and columns - Examples
# col1 = grid.columns[grid.columns < 0.3][-1]
# col2 = grid.columns[grid.columns >= 0.3][0]
# row1 = grid.index[grid.index < 110][-1]
# row2 = grid.index[grid.index >= 110][0]
# Nearest neighbours grid points
# [row1,col1], [row1, col2], [row2, col2], [row2, col1]
# Get option values
# V = [grid.loc[row1,col1], grid.loc[row1,col2], grid.loc[row2, col2], grid.loc[row2, col1]]
# Areas of the rectangle made by four corners and interior points
# A = [(row2-105) * (col2-0.3),
#      (row2-105) * (0.3-col1),
#      (105-row1) * (0.3-col1),
#      (105-row1) * (col2-0.3)]
# Option value in mesh points, approximated
# sum(array(V)*array(A))/sum(array(A))

## Convergence Analysis

In [24]:
# Iterate over asset steps (NAS)
nas_list = [10,20,30,40,50,60]
fdmoption = []

for i in nas_list:
    fdmoption.append(fdm_option(100,0.2,0.05,1,i).loc[100,1])
fdmoption

[9.51, 10.26, 10.37, 10.4, 10.42, 10.43]

In [25]:
# Call black scholes class
from Black_Scholes import BS

# Instantiate black scholes object
option = BS(100,100,0.05,1,0.20)
bsoption = round(option.callPrice,2)
bsoption = bsoption.repeat(len(nas_list))

# Range of option price
bsoption

array([10.45, 10.45, 10.45, 10.45, 10.45, 10.45])

In [26]:
# Subsume into dataframe
df = pd.DataFrame(list(zip(bsoption,fdmoption)), columns=['BS', 'FDM'],index=nas_list)
df['dev'] = df['FDM'] - df['BS']
df['% dev'] = round(df['dev'] / df['BS'] * 100.,2)

# Output
print("BS - FDM Convergence over NAS")
df

BS - FDM Convergence over NAS


,BS,FDM,dev,% dev
10,10.45,9.51,-0.94,-9.00
20,10.45,10.26,-0.19,-1.82
30,10.45,10.37,-0.08,-0.77
40,10.45,10.40,-0.05,-0.48
50,10.45,10.42,-0.03,-0.29
60,10.45,10.43,-0.02,-0.19
